# Prepare prompts

#### 1. Dependencies

In [15]:
import pandas as pd

#### 2. Settings

In [16]:
TRANSCRIPTS_PATH = "../ted_talks_transcripts.tsv"
TEMPLATE_PATH = "./prompt_template.txt"
OUTPUT_PATH = "./prompts.tsv"

TARGET_LANGUAGES = {
    "zh": "Chinese",
    "fi": "Finnish",
    "fr": "French",
    "he": "Hebrew",
    "ja": "Japanese",
    "pl": "Polish",
}

#### 3. Load transcripts and template

In [17]:
transcripts = pd.read_csv(TRANSCRIPTS_PATH, sep="	")
template = open(TEMPLATE_PATH, encoding="utf-8").read()

print(f"Wczytano {len(transcripts)} talkow")
print(f"Template: {len(template)} znakow")

Wczytano 277 talkow
Template: 2198 znakow


#### 4. Build prompts

In [18]:
def build_prompt(template: str, lang_name: str, transcript: str) -> str:
    return template.replace("[LANGUAGE]", lang_name).replace("[TRANSCRIPT]", transcript)


rows = []
for _, talk in transcripts.iterrows():
    en_text = talk.get("en_timed")
    if not isinstance(en_text, str) or not en_text.strip():
        continue
    slug = talk["slug"]
    for lang_code, lang_name in TARGET_LANGUAGES.items():
        rows.append({
            "id": f"{lang_code}--{slug}",
            "slug": slug,
            "target_lang": lang_code,
            "target_lang_name": lang_name,
            "prompt": build_prompt(template, lang_name, en_text),
        })

df = pd.DataFrame(rows)
print(f"Lacznie promptow: {len(df)}")
print(f"Per jezyk: {df['target_lang'].value_counts().to_dict()}")

Lacznie promptow: 1662
Per jezyk: {'zh': 277, 'fi': 277, 'fr': 277, 'he': 277, 'ja': 277, 'pl': 277}


#### 5. Save

In [19]:
df.to_csv(OUTPUT_PATH, sep="	", index=False)
print(f"Zapisano -> {OUTPUT_PATH}")

Zapisano -> ./prompts.tsv


#### 6. Display

In [20]:
df.head(3)

,id,slug,target_lang,target_lang_name,prompt
0,zh--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,zh,Chinese,You are a professional translator. Translate t...
1,fi--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,fi,Finnish,You are a professional translator. Translate t...
2,fr--adam_grant_how_to_stop_languishing_and_sta...,adam_grant_how_to_stop_languishing_and_start_f...,fr,French,You are a professional translator. Translate t...


In [21]:
print(df.iloc[0]["prompt"][:1500])

You are a professional translator. Translate the following English TED Talk
transcript into Chinese.

The transcript is in SRT subtitle format. Each cue block consists of:
  1. a cue number
  2. a timestamp line (HH:MM:SS,mmm --> HH:MM:SS,mmm)
  3. one or more lines of text

Rules:
- Preserve every cue number exactly as it appears
- Preserve every timestamp line exactly as it appears
- Translate only the text lines of each cue
- Return the full translated transcript in the same SRT format
- Where appropriate, choose informal, colloquial terms over formal or academic ones.
- Choose modern terms and phrases over traditional ones. Translators should be well-versed in the topics covered.
- Strive to match the tone and flow of the speaker's original talk. Rather than produce a word-for-word translation, aim to find the color, energy and "poetry" in the speaker's organic style and try to emulate it.
- Choose words and phrases that are most universally understood among all dialects.
- Instead